# Preprocesamiento del conjunto de datos del proyecto

Curso: Minería de datos

Grupo: 04

Proyecto: Clasificación automática de emociones en publicaciones de jóvenes peruanos mediante técnicas de minería de datos.

## Preparación del entorno

In [ ]:
# Instalación de librerías necesarias

!pip install pandas spacy

Descargar el modelo en español de `spaCy`, ya que contiene vocabulario en español, reglas lingüísticas, lematizador y etiquetador gramatical.

In [ ]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 106.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#### Importación de librerías

In [ ]:
import pandas as pd
import re
import spacy

#### Cargar el modelo Spacy

In [ ]:
nlp = spacy.load("es_core_news_sm")

#### Conexión con Google Drive y visualización del dataset

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

def cargar_dataset(ruta):

  try:
      df = pd.read_csv(ruta, sep=';', lineterminator='\n', on_bad_lines='skip')
      return df
  except Exception as e:
      print(f"Error al intentar cargar el archivo: {e}")
      return None

ruta_dataset = "/content/drive/MyDrive/Proyecto-Mineria-G4/Fuentes-Informacion"

df_youtube_01 = cargar_dataset(f"{ruta_dataset}/scrapping-youtube/comentarios_clasificado_parte01.csv")
df_youtube_02 = cargar_dataset(f"{ruta_dataset}/scrapping-youtube/comentarios_clasificado_parte02.csv")
df_tiktok_01 = cargar_dataset(f"{ruta_dataset}/scrapping-tiktok/comentarios_clasificado_parte01.csv")

# Eliminar las "filas fantasma" vacías ---
# Le decimos a Pandas: "Si la columna del comentario está vacía (NaN), borra toda la fila"
df_youtube_01 = df_youtube_01.dropna(subset=['content'])
df_youtube_02 = df_youtube_02.dropna(subset=['content'])
df_tiktok_01 = df_tiktok_01.dropna(subset=['Content'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Fase 01: Preprocesamiento

Visualización de los conjuntos de datos

In [ ]:
# Función para visualizar el dataset y sus dimensiones
# Función para visualizar el dataset y sus dimensiones
def visualizar_dataset(df):
    print("\n--- Dimensiones ---")
    print(df.shape)

    print("\n--- Primeras 5 filas ---")
    print(df.head(5))

    print("\n--- Valores nulos ---")
    print(df.isnull().sum().to_string())

In [ ]:
# Youtube parte 1
visualizar_dataset(df_youtube_01)


--- Dimensiones ---
(6031, 7)

--- Primeras 5 filas ---
          source_name                                   source_url  \
0  La República - LR+  https://www.youtube.com/watch?v=9Tfkd6aIqqI   
1     CLAUDIO REVATTA  https://www.youtube.com/watch?v=QrE4aGbUGAE   
2     CLAUDIO REVATTA  https://www.youtube.com/watch?v=QrE4aGbUGAE   
3     CLAUDIO REVATTA  https://www.youtube.com/watch?v=QrE4aGbUGAE   
4     CLAUDIO REVATTA  https://www.youtube.com/watch?v=QrE4aGbUGAE   

  content_type                                            content  \
0      comment  Increíble, Pacto mafioso, ahora quieren apoder...   
1      comment            La universidad del pacífico, la uni...😂   
2      comment  Con eso lo de la presentación de la virgen Mar...   
3      comment       Jajaja de universidades se metió el chamo :v   
4      comment  Falto universidad cientifica del SUR con los e...   

         date_published language   emocion\r  
0  2026-05-13T21:49:16Z       es       Ira\r  
1  2025-05-02

In [ ]:
# Youtube parte 2
visualizar_dataset(df_youtube_02)


--- Dimensiones ---
(4276, 7)

--- Primeras 5 filas ---
        source_name                                   source_url content_type  \
0  AprendemosJuntos  https://www.youtube.com/watch?v=2eQGcYotnp0      comment   
1  AprendemosJuntos  https://www.youtube.com/watch?v=2eQGcYotnp0      comment   
2  AprendemosJuntos  https://www.youtube.com/watch?v=2eQGcYotnp0      comment   
3  AprendemosJuntos  https://www.youtube.com/watch?v=2eQGcYotnp0      comment   
4  AprendemosJuntos  https://www.youtube.com/watch?v=2eQGcYotnp0      comment   

                                             content        date_published  \
0  Gracias por todo, en momentos de angustia, ans...  2024-11-19T02:24:57Z   
1  El corazón se me acelero cuando hablo del tras...  2021-10-13T21:03:37Z   
2  Que bueno es ver una maestra interesada en la ...  2024-04-19T02:42:06Z   
3  Cuidar de tu salud mental te permitirá vivir u...  2024-07-04T23:32:22Z   
4  Muchas gracias por su conferencia Dra gracias ...  2021-10-12T1

In [ ]:
# Tiktok
visualizar_dataset(df_tiktok_01)


--- Dimensiones ---
(130, 5)

--- Primeras 5 filas ---
                  Name                                         Comment URL  \
0   Angie Durand Flores  https://www.tiktok.com/@angiedurandflores5/vid...   
1  Isaac Yluquiz Contre  https://www.tiktok.com/@pedrito598/video/73540...   
2               skudexa  https://www.tiktok.com/@skudexa0/video/7354011...   
3                 Ándre  https://www.tiktok.com/@andre_notfound/video/7...   
4            𝓘𝓶𝓮𝓵𝓲𝓼𝓼𝓪 🌙  https://www.tiktok.com/@imelissabel/video/7354...   

                                             Content                 Date  \
0  la depresión es una enfermedad que no se va, s...  2024-04-04 14:57:30   
1  estoy dejando mi vida en la ruina no tengo nec...  2024-04-04 15:20:32   
2                                                🥺🥺🥺  2024-04-04 15:37:51   
3                          Cuando no se siente nada?  2024-04-04 16:19:49   
4  Dime si está bien solo querer dormir 🛌 y llora...  2024-04-22 18:35:59   

    emocion\

### Limpieza de texto

Realizaremos la limpieza del texto de los comentarios del dataframe de Youtube y Tiktok.
Lo que haremos será lo siguiente:

*   Conversión a minúsculas.
*   Eliminación de URLs.
*   Eliminación de caracteres especiales.
*   Eliminación de espacios múltiples.
*   Elimina URLs.
*   Elimina menciones (@usuario).
*   Elimina hashtags (#tema).
*   Elimina caracteres especiales.
*   Elimina espacios múltiples.

Se conservan los emojis porque ayudan a distinguir emociones.

In [ ]:
def limpiar_texto(texto):
    """
    Limpieza de comentarios de redes sociales:
    - Convierte a minúsculas.
    - Elimina URLs.
    - Elimina menciones (@usuario).
    - Elimina hashtags (#tema).
    - Conserva emojis.
    - Elimina caracteres especiales.
    - Elimina espacios múltiples.
    """

    # Manejo de valores nulos
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = texto.lower()

    # Eliminar URLs
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)

    # Eliminar menciones
    texto = re.sub(r'@\w+', '', texto)

    # Eliminar hashtags
    texto = re.sub(r'#\w+', '', texto)

    # Eliminar caracteres especiales
    # Conservando letras, espacios y emojis
    texto = re.sub(
        r'[^a-záéíóúñü\s'
        r'\U0001F300-\U0001FAFF'
        r'\U00002600-\U000027BF]',
        ' ',
        texto
    )

    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [ ]:
# Para los dataframes de Youtube
df_youtube_01['contenido_limpio'] = df_youtube_01['content'].apply(limpiar_texto)
df_youtube_02['contenido_limpio'] = df_youtube_02['content'].apply(limpiar_texto)

# Para el dataframe de tiktok
df_tiktok_01['contenido_limpio'] = df_tiktok_01['Content'].apply(limpiar_texto)

Ahora realizaremos la comparación entre el antes y después de aplicar la limpieza del texto.

In [ ]:
def limpieza_antes_despues(df, contenido, n=5):
    comparacion = pd.DataFrame({
        "Original": df[contenido].head(n),
        "Limpio": df["contenido_limpio"].head(n)
    })

    return comparacion

In [ ]:
print("Comparación de la limpieza del texto de los dataframes\n")
print("--- Youtube - Parte 01 ---\n")
print(limpieza_antes_despues(df_youtube_01, "content"))

print("\n--- Youtube - Parte 02 ---\n")
print(limpieza_antes_despues(df_youtube_02, "content"))

print("\n--- Tiktok ---\n")
print(limpieza_antes_despues(df_tiktok_01, "Content"))

Comparación de la limpieza del texto de los dataframes

--- Youtube - Parte 01 ---

                                            Original  \
0  Increíble, Pacto mafioso, ahora quieren apoder...   
1            La universidad del pacífico, la uni...😂   
2  Con eso lo de la presentación de la virgen Mar...   
3       Jajaja de universidades se metió el chamo :v   
4  Falto universidad cientifica del SUR con los e...   

                                              Limpio  
0  increíble pacto mafioso ahora quieren apoderar...  
1               la universidad del pacífico la uni 😂  
2  con eso lo de la presentación de la virgen mar...  
3        jajaja de universidades se metió el chamo v  
4  falto universidad cientifica del sur con los e...  

--- Youtube - Parte 02 ---

                                            Original  \
0  Gracias por todo, en momentos de angustia, ans...   
1  El corazón se me acelero cuando hablo del tras...   
2  Que bueno es ver una maestra interesada en la ...

### Tokenización

Realizaremos la tokenización para dividir los comentarios en palabras individuales.

In [ ]:
def tokenizar_spacy(texto):
    doc = nlp(texto)
    return [token.text for token in doc]

In [ ]:
print("Tokenización de los comentarios\n")
print("--- Youtube - Parte 01 ---\n")
df_youtube_01['tokens'] = df_youtube_01['contenido_limpio'].apply(tokenizar_spacy)
print(df_youtube_01['tokens'].head(5))

print("\n--- Youtube - Parte 02 ---\n")
df_youtube_02['tokens'] = df_youtube_02['contenido_limpio'].apply(tokenizar_spacy)
print(df_youtube_02['tokens'].head(5))

print("\n--- Tiktok ---\n")
df_tiktok_01['tokens'] = df_tiktok_01['contenido_limpio'].apply(tokenizar_spacy)
print(df_tiktok_01['tokens'].head(5))

Tokenización de los comentarios

--- Youtube - Parte 01 ---

0    [increíble, pacto, mafioso, ahora, quieren, ap...
1         [la, universidad, del, pacífico, la, uni, 😂]
2    [con, eso, lo, de, la, presentación, de, la, v...
3    [jajaja, de, universidades, se, metió, el, cha...
4    [falto, universidad, cientifica, del, sur, con...
Name: tokens, dtype: object

--- Youtube - Parte 02 ---

0    [gracias, por, todo, en, momentos, de, angusti...
1    [el, corazón, se, me, acelero, cuando, hablo, ...
2    [que, bueno, es, ver, una, maestra, interesada...
3    [cuidar, de, tu, salud, mental, te, permitirá,...
4    [muchas, gracias, por, su, conferencia, dra, g...
Name: tokens, dtype: object

--- Tiktok ---

0    [la, depresión, es, una, enfermedad, que, no, ...
1    [estoy, dejando, mi, vida, en, la, ruina, no, ...
2                                            [🥺, 🥺, 🥺]
3                       [cuando, no, se, siente, nada]
4    [dime, si, está, bien, solo, querer, dormir, 🛌...
Name: tokens

No se esta considerando la etapa de `eliminación de stopwords` debido a que muchos términos considerados stopwords en español, así como negaciones y adverbios de frecuencia, pueden aportar información para la identificación de estados emocionales.

### Lematización

Reduciremos las palabras a su forma base.

In [ ]:
def lematizar_tokens(tokens):

    texto = " ".join(tokens)

    doc = nlp(texto)

    lemas = []

    for token in doc:

        # Mantener emojis
        if token.is_punct:
            continue

        lema = token.lemma_.strip()

        if lema:
            lemas.append(lema)

    return lemas

In [ ]:
print("Lematización de las palabras a su forma base")
print("--- Youtube - Parte 01 ---\n")
df_youtube_01['lemas'] = (df_youtube_01['tokens'].apply(lematizar_tokens))
print(df_youtube_01['lemas'].head(5))

print("\n--- Youtube - Parte 02 ---\n")
df_youtube_02['lemas'] = (df_youtube_02['tokens'].apply(lematizar_tokens))
print(df_youtube_02['lemas'].head(5))

print("\n--- Tiktok ---\n")
df_tiktok_01['lemas'] = (df_tiktok_01['tokens'].apply(lematizar_tokens))
print(df_tiktok_01['lemas'].head(5))

Lematización de las palabras a su forma base
--- Youtube - Parte 01 ---

0    [increíble, pacto, mafioso, ahora, querer, apo...
1         [el, universidad, del, pacífico, el, uni, 😂]
2    [con, ese, él, de, el, presentación, de, el, v...
3    [jajaja, de, universidad, él, meter, el, chamo...
4    [faltar, universidad, cientifico, del, sur, co...
Name: lemas, dtype: object

--- Youtube - Parte 02 ---

0    [gracias, por, todo, en, momento, de, angustio...
1    [el, corazón, él, yo, acelerar, cuando, hablar...
2    [que, bueno, ser, ver, uno, maestra, interesad...
3    [cuidar, de, tu, salud, mental, tú, permitir, ...
4    [mucho, gracia, por, su, conferencia, dra, gra...
Name: lemas, dtype: object

--- Tiktok ---

0    [el, depresión, ser, uno, enfermedad, que, no,...
1    [estar, dejar, mi, vida, en, el, ruina, no, te...
2                                            [🥺, 🥺, 🥺]
3                       [cuando, no, él, sentir, nada]
4    [dime, si, estar, bien, solo, querer, dormir, ...
Na

### Verificación del preprocesamiento del texto

In [ ]:
print("Verificación del preprocesamiento de texto")
print("--- Youtube - Parte 01 ---\n")
print(df_youtube_01[['contenido_limpio', 'tokens', 'lemas']].head(5))

print("\n--- Youtube - Parte 02 ---\n")
print(df_youtube_02[['contenido_limpio', 'tokens', 'lemas']].head(5))

print("\n--- Tiktok ---\n")
print(df_tiktok_01[['contenido_limpio', 'tokens', 'lemas']].head(5))

Verificación del preprocesamiento de texto
--- Youtube - Parte 01 ---

                                    contenido_limpio  \
0  increíble pacto mafioso ahora quieren apoderar...   
1               la universidad del pacífico la uni 😂   
2  con eso lo de la presentación de la virgen mar...   
3        jajaja de universidades se metió el chamo v   
4  falto universidad cientifica del sur con los e...   

                                              tokens  \
0  [increíble, pacto, mafioso, ahora, quieren, ap...   
1       [la, universidad, del, pacífico, la, uni, 😂]   
2  [con, eso, lo, de, la, presentación, de, la, v...   
3  [jajaja, de, universidades, se, metió, el, cha...   
4  [falto, universidad, cientifica, del, sur, con...   

                                               lemas  
0  [increíble, pacto, mafioso, ahora, querer, apo...  
1       [el, universidad, del, pacífico, el, uni, 😂]  
2  [con, ese, él, de, el, presentación, de, el, v...  
3  [jajaja, de, universidad, él, me

## Exportar dataframes

In [ ]:
def exportar_excel(df, nombre_archivo):
    """
    Exporta un DataFrame a un archivo Excel.
    """

    try:
        df.to_excel(nombre_archivo, index=False)
        print(f"Dataset exportado correctamente: {nombre_archivo}")

    except Exception as e:
        print(f"Error al exportar el archivo: {e}")

In [ ]:
exportar_excel(df_youtube_01, "dataset_youtube_01_procesado.xlsx")
exportar_excel(df_youtube_02, "dataset_youtube_02_procesado.xlsx")
exportar_excel(df_tiktok_01, "dataset_tiktok_01_procesado.xlsx")

Dataset exportado correctamente: dataset_youtube_01_procesado.xlsx
Dataset exportado correctamente: dataset_youtube_02_procesado.xlsx
Dataset exportado correctamente: dataset_tiktok_01_procesado.xlsx
